# Control: linearise, place the poles, run the closed loop, compare with LQR

Take the Morris-Lecar cell with the `hopf` parameter set and `iapp = 0.3`.
Its equilibrium is unstable (see `02_bifurcations.ipynb`): the voltage
oscillates. Can we use the applied current as a control input, `iapp = 0.3 -
K (x - x*)` with `x*` the equilibrium, to hold the cell at that equilibrium?
The steps are: linearise, check controllability, place the poles, run the
closed loop, and compare with the optimal (LQR) gain.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xpplab as xs

m = xs.load(xs.example("lecar.ode"))
m.apply_set("hopf")
m.parameters["iapp"] = 0.3
m.equilibrium()

## Linearise

`m.linearize(inputs=["iapp"])` linearises about the equilibrium and takes
`iapp` as the input: `A` is the Jacobian, `B` the derivative with respect
to the input.

In [ ]:
lin = m.linearize(inputs=["iapp"])
print("A =", lin.A, sep="\n")
print("B =", lin.B, sep="\n")
lin.eigenvalues

Both eigenvalues have a positive real part, so the equilibrium is unstable.

## Is it controllable?

The current enters the `V` equation only, but `W` follows `V`, so the input
reaches both states. The controllability matrix has full rank.

In [ ]:
lin.rank, lin.controllable

In [ ]:
lin.plot();

## Place the poles

`lin.place_poles` returns the gain `K` for which `A - B K` has the poles you
ask for. Choose a stable pair, `-0.5 +- 0.3j`.

In [ ]:
K = lin.place_poles([-0.5 + 0.3j, -0.5 - 0.3j])
K

In [ ]:
np.linalg.eigvals(lin.A - lin.B @ K)

## Run the closed loop

`lin.closed_loop(K)` writes the closed-loop model, the full nonlinear one
with `iapp` replaced by the feedback law, and loads it. Start both the
open and the closed loop away from the equilibrium.

In [ ]:
start = {"V": 0.0, "W": 0.4}

m.ics.update(start)
open_loop = m.run(total=60)

cl = lin.closed_loop(K)
cl.ics.update(start)
placed = cl.run(total=60)

fig, ax = plt.subplots()
ax.plot(open_loop.t, open_loop["V"], label="open loop")
ax.plot(placed.t, placed["V"], label="poles placed")
ax.set_xlabel("t"); ax.set_ylabel("V")
ax.legend();

In [ ]:
eq = m.equilibrium()
print("V at the equilibrium:      ", eq.values["V"])
print("V at the end, open loop:   ", open_loop["V"][-1])
print("V at the end, closed loop: ", placed["V"][-1])

Without feedback the cell keeps oscillating. With it, `V` settles at the
equilibrium. The `CUR` column of the closed loop is the current that the
feedback applied.

## Compare with LQR

LQR chooses the gain that minimises a cost, the integral of
`dx' Q dx + u' R u`, so you set weights instead of poles. A large `R` makes
the input expensive.

In [ ]:
res = lin.lqr(Q=[10, 10], R=1)
res.K, res.poles

In [ ]:
cl_lqr = lin.closed_loop(res.K)
cl_lqr.ics.update(start)
optimal = cl_lqr.run(total=60)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, col in [(a1, "V"), (a2, "CUR")]:
    ax.plot(placed.t, placed[col], label="poles placed")
    ax.plot(optimal.t, optimal[col], label="LQR")
    ax.set_xlabel("t"); ax.set_ylabel(col)
    ax.legend();

In [ ]:
for name, run in [("poles placed", placed), ("LQR", optimal)]:
    print(name, "peak |iapp - 0.3|:", np.abs(run["CUR"] - 0.3).max())

Both gains stabilise the cell. Here the LQR design asks for a much larger
peak current than the placed poles, and one of its poles (-0.34) is slower
than the placed pair. LQR does not aim at chosen poles; it balances the
state error against the input you spend. Change `Q` and `R` and run the
cells again to see the trade-off move.

## Where next

- [Control design](https://cesarperdomodiaz.github.io/xpplab/python/control/)
  covers `valid_radius` (how far the linear design can be trusted), the root
  locus and models with several inputs.